# 精簡搜尋：可重現Colab
會重新訓練四條有界路線，不保證Kaggle0.680；僅通過兩種子門檻才產生候選。
請從分支research-0680-20261024的Code → Download ZIP取得完整儲存庫，再於下方上傳。若只用第三輪研究包，另上傳train.csv、test.csv、sample_submission.csv。私人repo先在瀏覽器登入下載，不需Token。


In [ ]:
from pathlib import Path
import os,sys,subprocess,json,zipfile,time,shutil
base=Path.cwd()
matches=[p.parent for p in base.rglob('train.csv') if (p.parent/'research_0680'/'src').is_dir() and '.venvs' not in p.parts]
if not matches:
    try:
        from google.colab import files
    except ImportError:
        raise RuntimeError('請在既有儲存庫目錄執行，或在 Colab 上傳完整儲存庫 ZIP。')
    uploaded=files.upload()
    target=base/'earthquake_repository';target.mkdir(exist_ok=True)
    for name in uploaded:
        if name.endswith('.zip'):
            with zipfile.ZipFile(name) as archive:
                for item in archive.infolist():
                    destination=(target/item.filename).resolve()
                    if not destination.is_relative_to(target.resolve()):
                        raise ValueError('拒絕 ZIP 路徑越界')
                archive.extractall(target)
    for project in target.rglob('research_0680'):
        if not (project/'src').is_dir():continue
        for name in ['train.csv','test.csv','sample_submission.csv']:
            if name in uploaded and not (project.parent/name).exists():
                shutil.copyfile(base/name,project.parent/name)
    matches=[p.parent for p in target.rglob('train.csv') if (p.parent/'research_0680'/'src').is_dir()]
if not matches:
    raise RuntimeError('找不到同時含 train.csv 與 research_0680/src 的完整儲存庫。請使用 GitHub Code → Download ZIP。')
repo=matches[0].resolve()
assert all((repo/f).is_file() for f in ['train.csv','test.csv','sample_submission.csv'])
print('Repository:',repo)


In [ ]:
subprocess.check_call([sys.executable,'-m','pip','install','-r',str(repo/'research_0680/configs/requirements.txt')])
for name in ['OPENBLAS_NUM_THREADS','OMP_NUM_THREADS','MKL_NUM_THREADS']:
    os.environ[name]='1'
run_dir=repo/'research_0680'/'phase3_colab_runs'/time.strftime('%Y%m%d_%H%M%S')
run_dir.mkdir(parents=True,exist_ok=False)
os.environ['RESEARCH0680_RESULTS']=str(run_dir/'results')
os.environ['RESEARCH0680_REPORTS']=str(run_dir/'reports')
for envname,subdir in [('RESEARCH0680_PHASE3_RESULTS','trees_spline'),('RESEARCH0680_PHASE3_BALANCED_RESULTS','balanced'),('RESEARCH0680_PHASE3_INTERACTION_RESULTS','interaction'),('RESEARCH0680_PHASE3_REPAIR_RESULTS','repair')]:
    os.environ[envname]=str(run_dir/subdir)
os.environ['MPLCONFIGDIR']=str(run_dir/'matplotlib_cache')
print('Fresh output directory:',run_dir)


## 執行
沿用單一nested runner；每折選擇只看允許訓練資料。400棵淺樹及少量邏輯模型，不做大量搜尋。CPU可能需數分鐘至十多分鐘；每折保存結果。輸出寫入新目錄，不覆寫已交付檔案。


In [ ]:
for module in ['run','balanced','interactions','repair']:
    subprocess.check_call([sys.executable,'-B','-m','research_0680.phase3.'+module],cwd=repo,env=os.environ.copy())


In [ ]:
for folder in ['trees_spline','balanced','interaction','repair']:
    path=run_dir/folder
    print(folder,(path/'submission_gate.json').read_text())
    if (path/'nested_comparison.csv').exists():
        print((path/'nested_comparison.csv').read_text())


## 下載
本次既有結果沒有通過門檻的新提交。重跑如亦未通過，沒有submission_candidate檔是預期行為；不能用開發切分最高值代替Kaggle分數。


In [ ]:
bundle=base/'Kaggle_Phase3_reproduction_results.zip'
with zipfile.ZipFile(bundle,'w',compression=zipfile.ZIP_DEFLATED) as z:
    for p in run_dir.rglob('*'):
        if p.is_file() and 'matplotlib_cache' not in p.parts:
            z.write(p,p.relative_to(run_dir))
print(bundle,bundle.stat().st_size)
try:
    from google.colab import files
    files.download(str(bundle))
except ImportError:
    print('本機結果位於上面列出的 ZIP 路徑。')
